In [1]:
import os
import OpenDartReader
from dotenv import load_dotenv

load_dotenv()
api_key = os.getenv('DART_API')
dart = OpenDartReader(api_key)

In [ ]:
import FinanceDataReader as fdr
import pandas as pd

dart_all = dart.corp_codes
krx_kosdaq = fdr.StockListing('KOSDAQ') # krx_kosdaq 코스닥 다 가져오기
krx_kosdaq = krx_kosdaq[(krx_kosdaq['Dept'] != 'SPAC(소속부없음)') & (~krx_kosdaq['Name'].str.contains('호스팩'))] # SPAC이랑 이름에 '호스팩' 들어간 불필요한 기업 제거

# 데이터 전처리 (타입 맞추기 및 결측치 제거)
dart_all = dart_all[dart_all['stock_code'].notnull()].copy() # notnull이 없긴한데, 예방차원
dart_all['stock_code'] = dart_all['stock_code'].astype(str).str.zfill(6) # stock_code가 비어있는 행 000000으로 변경, 6자리 문자열 변환 (6자리로 뜨긴하는데 혹시 모르니까 0 채우기)

# KRX의 'Code' 컬럼을 6자리 문자열 변환(6자리긴한데 혹시 모르니깐 0 채우기)
krx_kosdaq['Code'] = krx_kosdaq['Code'].astype(str).str.zfill(6)

# DART의 stock_code와 KRX의 Code를 기준으로 병합 (dart기업이름, dart기업코드, dart주식코드, krx주식코드, krx기업이름)
kosdaq_df = pd.merge(
    dart_all[['corp_name', 'corp_code', 'stock_code']], 
    krx_kosdaq[['Code', 'Name']], 
    right_on='Code', 
    left_on='stock_code', 
    how='inner'
)

kosdaq_df = kosdaq_df.drop('Code', axis=1) # 주식코드는 중복이니깐 삭제

print(f"매칭된 코스닥 기업 수: {len(kosdaq_df)}개")
print(kosdaq_df.head())

kosdaq_df.to_csv('kosdaq_corp_map.csv', index=False, encoding='utf-8-sig')

매칭된 코스닥 기업 수: 1738개
  corp_name corp_code stock_code     Name
0       GRT  01170962     900290      GRT
1       로스웰  01139266     900260      로스웰
2   크리스탈신소재  01107665     900250  크리스탈신소재
3      인화정공  00482426     101930     인화정공
4      대원산업  00111874     005710     대원산업


In [31]:
merge_result = pd.merge(
    dart_all[['corp_name', 'corp_code', 'stock_code']], 
    krx_kosdaq[['Code', 'Name']], 
    right_on='Code', 
    left_on='stock_code', 
    how='outer',
    indicator=True
)
only_krx = merge_result[merge_result['_merge'] == 'right_only']


print(f"KRX에만 있는 기업(DART 매칭 실패): {len(only_krx)}개")
only_krx
# 대호특수강, 소프트센, 해성산업으로 dart는 존재하니깐 굳이 없어도 될듯
# 일단 1819개 중에서 spac이랑 기타 등등 처음부터 제외할 코스닥 기업(페이퍼 컴퍼니) 있는지 논의 후 제거하기

KRX에만 있는 기업(DART 매칭 실패): 3개


,corp_name,corp_code,stock_code,Code,Name,_merge
112067,NaN,NaN,NaN,021045,대호특수강우,right_only
112255,NaN,NaN,NaN,032685,소프트센우,right_only
112323,NaN,NaN,NaN,03481K,해성산업1우,right_only


In [32]:
import pandas as pd

df = pd.read_csv('kosdaq_corp_map.csv', dtype={'corp_code': str, 'stock_code': str})

corp_code_list = df['corp_code'].tolist()

print(f"추출된 고유번호 개수: {len(corp_code_list)}개")

추출된 고유번호 개수: 1738개


In [15]:
# dart의 기업코드로 산업분류(2자리로만)
counts = {}
matches = {}
missing_codes = [] # 조회가 안 된 종목들을 따로 담을 리스트

for i in corp_code_list:
    try:
        company_info = dart.company(i)
        industry_code = company_info['induty_code'][:2]
        
        if industry_code:
            # 딕셔너리 카운팅 로직
            if industry_code in counts:
                counts[industry_code] += 1
                matches[industry_code].append(company_info['stock_name'])
            else:
                counts[industry_code] = 1
                matches[industry_code] = [company_info['stock_name']]
        else:
            # industry_code가 None이거나 빈 값인 경우 출력
            print(f" 업종 코드 없음: {company_info['stock_name']}")
            missing_codes.append(company_info['stock_name'])
                
    except Exception as e:
        # API 호출 자체가 실패한 경우 (네트워크 에러, 존재하지 않는 번호 등) dart.company의 오류 방지용
        print(f"조회 에러 발생 ({company_info['stock_name']}): {e}")
        missing_codes.append(company_info['stock_name'])

print("-" * 30)
print("최종 결과:", counts)
print("누락된 종목 총 개수:", len(missing_codes))

------------------------------
최종 결과: {'64': 38, '31': 19, '30': 58, '17': 8, '29': 185, '41': 17, '24': 34, '28': 61, '58': 181, '25': 41, '27': 96, '26': 252, '20': 97, '70': 62, '45': 3, '59': 33, '63': 24, '66': 5, '13': 5, '21': 135, '46': 79, '62': 30, '33': 12, '91': 1, '18': 2, '11': 6, '22': 29, '61': 9, '75': 10, '10': 42, '14': 15, '85': 9, '90': 5, '55': 2, '42': 11, '72': 13, '49': 2, '15': 2, '71': 22, '47': 21, '73': 17, '52': 2, '23': 15, '32': 3, '16': 2, '68': 3, '35': 3, '60': 9, '38': 3, '01': 2, '95': 1, '74': 1, '56': 1}
누락된 종목 총 개수: 0


In [16]:
counts_df = pd.DataFrame(list(counts.items()), columns=['industry_code', 'count'])
counts_df = counts_df.sort_values(by='industry_code', ascending=True)
counts_df.to_csv('industry_counts.csv', index=False, encoding='utf-8-sig')


matches_readable = {k: ", ".join(v) for k, v in matches.items()}
matches_df = pd.DataFrame(list(matches_readable.items()), columns=['industry_code', 'corp_names'])
matches_df = matches_df.sort_values(by='industry_code', ascending=True)
matches_df.to_csv('industry_matches.csv', index=False, encoding='utf-8-sig')

In [ ]:
import pandas as pd
import FinanceDataReader as fdr
from tqdm import tqdm

file_path = 'kosdaq_corp_map.csv'
df = pd.read_csv(file_path, dtype={'corp_code': str, 'stock_code': str})

try:
    ind_map = pd.read_csv('industry_matches.csv')
    ind_dict = {name.strip(): str(r['industry_code']) for _, r in ind_map.iterrows() for name in str(r['corp_names']).split(',')}
except Exception as e:
    print(f"업종 매칭 파일(industry_matches.csv) 로드 중 오류 발생: {e}")
    ind_dict = {}

for col in ['est_dt', 'ipo', 'ind_code']:
    if col not in df.columns:
        df[col] = ""
    else:
        df[col] = df[col].astype(str)

print("거래소 상장 정보(KRX-DESC) 불러오기")
try:
    df_krx = fdr.StockListing('KRX-DESC')
    listing_map = df_krx.set_index('Code')['ListingDate'].dt.strftime('%Y-%m-%d').to_dict()
except Exception as e:
    print(f"거래소 데이터를 가져오는 중 오류 발생: {e}")
    listing_map = {}

print("상장일(ipo), 설립일(est_dt), 업종코드(ind_code) 매칭 시작")

for i, row in tqdm(df.iterrows(), total=len(df)):
    d_code = str(row['corp_code']).zfill(8)
    s_code = str(row['stock_code']).zfill(6)
    c_name = row['corp_name']
    
    df.at[i, 'ind_code'] = ind_dict.get(c_name, "")
    
    if s_code in listing_map:
        df.at[i, 'ipo'] = listing_map[s_code]
    
    try:
        info = dart.company(d_code)
        if info and 'est_dt' in info:
            raw_date = str(info['est_dt']) 
            if len(raw_date) == 8:
                df.at[i, 'est_dt'] = f"{raw_date[:4]}-{raw_date[4:6]}-{raw_date[6:]}"
            else:
                df.at[i, 'est_dt'] = raw_date
    except:
        continue

# 열 순서 재배치
new_columns_order = ['stock_code', 'corp_code', 'corp_name', 'est_dt', 'ipo', 'ind_code']
df_final = df[[c for c in new_columns_order if c in df.columns]]

output_name = 'kosdaq_corp_map_final.csv'
df_final.to_csv(output_name, index=False, encoding='utf-8-sig')

print(f"\n작업 완료! 최종 파일이 생성되었습니다: {output_name}")

In [ ]:
import pandas as pd

df = pd.read_csv('kosdaq_corp_map_final.csv', dtype=str)
df